In [16]:
import os
from feast import FeatureStore
import pandas as pd
from datetime import datetime

In [17]:
feature_store_path = os.path.join("feature_store", "feature_repo")

In [26]:
# Initialize the feature store
store = FeatureStore(repo_path=feature_store_path)

In [ ]:
print(store.list_feature_views())

In [28]:
# 1. Get historical features for training
entity_df = pd.DataFrame({
    "driver_id": [1001, 1005],
    "event_timestamp": [datetime(2024,10,17,12,0), datetime(2024, 10, 2, 12, 0)],
})
features=[
        "driver_conversion_stats:conv_rate",
        "driver_conversion_stats:acc_rate",
        "driver_activity_stats:avg_daily_trips",
        "realtime_driver_metrics:conv_rate_per_trip",
        "realtime_driver_metrics:adjusted_conv_rate",
]

In [29]:
entity_df

,driver_id,event_timestamp
0,1001,2024-10-17 12:00:00
1,1005,2024-10-02 12:00:00


In [30]:
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=features,
).to_df()

print("Training Data:\n")
print(training_df.info())

Training Data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   driver_id           2 non-null      int64              
 1   event_timestamp     2 non-null      datetime64[ns, UTC]
 2   conv_rate           2 non-null      float32            
 3   acc_rate            2 non-null      float32            
 4   avg_daily_trips     2 non-null      int32              
 5   conv_rate_per_trip  2 non-null      float64            
 6   adjusted_conv_rate  2 non-null      float32            
dtypes: datetime64[ns, UTC](1), float32(3), float64(1), int32(1), int64(1)
memory usage: 212.0 bytes
None


In [31]:
print(training_df.head())

   driver_id           event_timestamp  conv_rate  acc_rate  avg_daily_trips  \
0       1005 2024-10-02 12:00:00+00:00   0.230119  0.642878              551   
1       1001 2024-10-17 12:00:00+00:00   0.295067  0.257097              456   

   conv_rate_per_trip  adjusted_conv_rate  
0            0.000418            0.294407  
1            0.000647            0.320776  


In [35]:
# 2. Get online features for inference
online_features = store.get_online_features(
    features=[
        "driver_conversion_stats:conv_rate",
        "driver_conversion_stats:acc_rate",
        "driver_activity_stats:avg_daily_trips",
    ],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1005}],
).to_df()

print("Online features:")
print(online_features)

Online features:
   driver_id conv_rate acc_rate avg_daily_trips
0       1001      None     None            None
1       1005      None     None            None
